In [1]:
# Import libraries and configure Ollama

import pprint

from autogen import ConversableAgent, LLMConfig

MODEL = "llama3.2"

llm_config = LLMConfig({"model":MODEL, "api_type": "ollama", 
                        "client_host": "http://localhost:11434",},temperature = 0.3,)

print("AutoGen and Ollama configuration successful!")
print("Selected Model:", MODEL)

AutoGen and Ollama configuration successful!
Selected Model: llama3.2


In [2]:
# AutoGen agent

agent = ConversableAgent(name = "chatbot", llm_config = llm_config, human_input_mode = "NEVER",)

print("AutoGen agent created successfully!")

AutoGen agent created successfully!


In [3]:
# Generate a reply

reply = agent.generate_reply(messages = [{"content": "Tell me a joke.", "role": "user"}])

print(reply)

{'content': "Here's one:\n\nWhat do you call a fake noodle?\n\nAn impasta!", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


In [4]:
reply = agent.generate_reply(messages = [{"content": "Tell me a joke.","role": "user"},
                                         {"content": reply["content"],"role": "assistant"},
                                         {"content": "Repeat the joke.","role": "user"}])

print(reply)

{'content': "Here's the joke again:\n\nWhat do you call a fake noodle?\n\nAn impasta!", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


In [5]:
# Create two agents – Cathy and Joe

cathy = ConversableAgent(name = "cathy", system_message = ("Your name is Cathy and you are a stand-up comedian."),
                         llm_config = llm_config, human_input_mode = "NEVER")

joe = ConversableAgent(name = "joe", system_message = ("Your name is Joe and you are a stand-up comedian."
                                                      "Start the next joke from the punchline of the previous joke."),
                       llm_config = llm_config, human_input_mode = "NEVER")
        
print("Cathy and Joe agents created successfully!")

Cathy and Joe agents created successfully!


In [6]:
# Initiate conversation

chat_result = joe.initiate_chat(recipient=cathy, message="I'm Joe. Cathy, let's keep the jokes rolling.",
                                max_turns=2,)



joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Joe! Nice to meet you, buddy! So, I was walking down the street the other day, and I saw a guy with a shirt that said, "I'm not arguing, I'm just explaining why I'm right." I'm like, "Dude, that's not a shirt, that's a marriage certificate!" (laughs)

--------------------------------------------------------------------------------
joe (to cathy):

(laughs) Ahahahaha, nice one! You know, I was thinking about that guy the other day, and I realized he's probably married to his ex-wife... or at least, he's still trying to explain why he's right about who's the better spouse. (chuckles)

--------------------------------------------------------------------------------
cathy (to joe):

(laughs) Oh man, you're a genius, Joe! That's like taking a joke and turning it into a whole new level of comedy gold! I mean, can you imagine the the

In [7]:
# Print conversation history

pprint.pprint(chat_result.chat_history)

[{'content': "I'm Joe. Cathy, let's keep the jokes rolling.",
  'name': 'joe',
  'role': 'assistant'},
 {'content': 'Joe! Nice to meet you, buddy! So, I was walking down the street '
             'the other day, and I saw a guy with a shirt that said, "I\'m not '
             'arguing, I\'m just explaining why I\'m right." I\'m like, "Dude, '
             'that\'s not a shirt, that\'s a marriage certificate!" (laughs)',
  'name': 'cathy',
  'role': 'user'},
 {'content': '(laughs) Ahahahaha, nice one! You know, I was thinking about '
             "that guy the other day, and I realized he's probably married to "
             "his ex-wife... or at least, he's still trying to explain why "
             "he's right about who's the better spouse. (chuckles)",
  'name': 'joe',
  'role': 'assistant'},
 {'content': "(laughs) Oh man, you're a genius, Joe! That's like taking a joke "
             'and turning it into a whole new level of comedy gold! I mean, '
             'can you imagine the t

In [8]:
# Print conversation cost

pprint.pprint(chat_result.cost)

{'usage_excluding_cached_inference': {'llama3.2': {'completion_tokens': 228,
                                                   'cost': 0.0,
                                                   'prompt_tokens': 378,
                                                   'total_tokens': 606},
                                      'total_cost': 0.0},
 'usage_including_cached_inference': {'llama3.2': {'completion_tokens': 228,
                                                   'cost': 0.0,
                                                   'prompt_tokens': 378,
                                                   'total_tokens': 606},
                                      'total_cost': 0.0}}


In [9]:
pprint.pprint(chat_result.summary)

("(laughs) Oh man, you're a genius, Joe! That's like taking a joke and turning "
 'it into a whole new level of comedy gold! I mean, can you imagine the '
 'therapy sessions? "Honey, I\'m not arguing, I\'m just explaining why I\'m '
 'right... again." (laughs) And then the ex-wife is like, "Uh, no, dear, '
 'that\'s just code for \'I still think I\'m right\'!"')


In [10]:
# Get a better summary using an LLM

chat_result = joe.initiate_chat(recipient=cathy, message="I'm Joe. Cathy, let's keep the jokes rolling.",
                                max_turns=2, summary_method="reflection_with_llm",
                                summary_prompt="Summarize the conversation clearly and briefly.",)

print("Conversation Summary:")

pprint.pprint(chat_result.summary)

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Joe! Nice to meet you, buddy! So, I was walking down the street the other day, and I saw a guy with a shirt that said, "I'm not arguing, I'm just explaining why I'm right." I'm like, "Dude, that's not a shirt, that's a marriage certificate!" (laughs) What's new with you?

--------------------------------------------------------------------------------
joe (to cathy):

(laughs) Ahah, nice one! You know, I was thinking, if that guy's shirt is a marriage certificate, then his wife must be wearing a shirt that says "I'm not married, but I'll pretend to be for the rest of my life!" (laughs) And speaking of pretending, have you ever noticed how some people's Facebook profiles are just a series of carefully curated lies?

--------------------------------------------------------------------------------
cathy (to joe):

(laughs) Oh man

In [11]:
# Chat termination

def is_termination_message(msg):
    content = msg.get("content", "")

    if isinstance(content, str):

        return("I gotta go" in content
               or "Goodbye" in content)
    
    return False

In [12]:
# Create Cathy and Joe with termination conditions

cathy = ConversableAgent(name = "cathy",
                         system_message=("Your name is Cathy and you are a stand-up comedian."
                                         "Continue the conversation naturally."
                                         "When you are ready to end the conversation,"
                                         "say exactly 'I gotta go'."),
                         llm_config = llm_config, human_input_mode = "NEVER",
                         is_termination_msg = is_termination_message,)


joe = ConversableAgent(name = "joe", 
                       system_message = ("Your name is Joe and you are a stand-up comedian. "
                                         "Continue the conversation naturally. "
                                         "When you are ready to end the conversation, "
                                         "say exactly 'I gotta go'."),
                       llm_config = llm_config, human_input_mode = "NEVER",
                       is_termination_msg = is_termination_message,)

print("Agents created with termination conditions!")

Agents created with termination conditions!


In [13]:
# Initiate conversation with termination (We use termination conditions to tell an agent when to stop a conversation. 
# We use max_turns as a safety limit to prevent the agent from running forever.)

chat_result = joe.initiate_chat(recipient=cathy, 
                                message="I'm Joe. Cathy, let's keep the jokes rolling.")

print("Conversation completed!")

joe (to cathy):

I'm Joe. Cathy, let's keep the jokes rolling.

--------------------------------------------------------------------------------
cathy (to joe):

Joe! Nice to meet you, buddy! So, I was walking down the street the other day, and I saw a guy with a shirt that said, "I'm not arguing, I'm just explaining why I'm right." I'm like, "Dude, that's not a shirt, that's a marriage certificate!" (laughs)

--------------------------------------------------------------------------------
joe (to cathy):

(laughs) Oh man, that's amazing! I love it. You know, I've got one for you: Have you ever noticed how some restaurants put 'artisanal' on their menus to make it sound like the food was made by a local craftsman? "Artisanal toast, $12." No, dude, it's just toast. (laughs)

--------------------------------------------------------------------------------
cathy (to joe):

(laughs) Ahahaha, that's genius! I love it! You're absolutely right, though - if I'm paying $12 for toast, I expect t

In [14]:
# Display final conversation

print("Final Conversation History:")

pprint.pprint(chat_result.chat_history)

Final Conversation History:
[{'content': "I'm Joe. Cathy, let's keep the jokes rolling.",
  'name': 'joe',
  'role': 'assistant'},
 {'content': 'Joe! Nice to meet you, buddy! So, I was walking down the street '
             'the other day, and I saw a guy with a shirt that said, "I\'m not '
             'arguing, I\'m just explaining why I\'m right." I\'m like, "Dude, '
             'that\'s not a shirt, that\'s a marriage certificate!" (laughs)',
  'name': 'cathy',
  'role': 'user'},
 {'content': "(laughs) Oh man, that's amazing! I love it. You know, I've got "
             'one for you: Have you ever noticed how some restaurants put '
             "'artisanal' on their menus to make it sound like the food was "
             'made by a local craftsman? "Artisanal toast, $12." No, dude, '
             "it's just toast. (laughs)",
  'name': 'joe',
  'role': 'assistant'},
 {'content': "(laughs) Ahahaha, that's genius! I love it! You're absolutely "
             "right, though - if I'm pa

In [15]:
# Display final summary

print("Final Conversation Summary:")

pprint.pprint(chat_result.summary)

Final Conversation Summary:
("(laughs) Oh man, I'm loving this! The Yelp reviews are going to be "
 "priceless. And you know what's even better? If we start getting some "
 'hipsters who actually believe the "artisanal water" hype and start paying us '
 '$10 a glass for it. We\'ll be like, "Sorry, dude, that\'s just tap water '
 'with a sticker on it." (laughs)\n'
 '\n'
 'And can you imagine the Instagram posts? "Just sipped on my latest artisanal '
 "water purchase from @Joe'sHandcraftedWater. The 'handcrafted' label is so on "
 'point #artisanalwater #locallysourced #tapwater"\n'
 '\n'
 'We should totally start selling this stuff at music festivals and stuff. '
 'People will be like, "Oh, I\'ll take a glass of your handcrafted water for '
 '$5, please." And we can just be like, "Sure thing, dude. It\'s got the '
 '\'Authenticity Guarantee\' sticker on it." (laughs)\n'
 '\n'
 'You know what? I think this is going to be a hit. We should start a whole '
 'brand around this stuff. "Joe\'